# Evaluating the Ideal Chunk Size for a RAG System using LlamaIndex

> **Refreshed November 2026** — modernized for the modular `llama-index-core` 0.14
> architecture (`Settings` instead of `ServiceContext`, `llama_index.core.evaluation`)
> and the GPT-6 generation of models.

# **Introduction**

Retrieval-augmented generation (RAG) fuses the extensive retrieval capabilities of search systems with the LLM. When implementing a RAG system, one critical parameter that governs the system's efficiency and performance is the `chunk_size`. How does one discern the optimal chunk size for seamless retrieval? This is where LlamaIndex **Response Evaluation** comes in handy. In this notebook, we'll guide you through the steps to determine the best `chunk_size` using LlamaIndex's response evaluation module. If you're unfamiliar with it, review the [evaluation module guide](https://docs.llamaindex.ai/en/stable/module_guides/evaluating/) before proceeding.

## **Why Chunk Size Matters**

Choosing the right `chunk_size` influences the efficiency and accuracy of a RAG system in several ways:

1. **Relevance and Granularity**: A small `chunk_size`, like 128, yields more granular chunks. This granularity, however, presents a risk: vital information might not be among the top retrieved chunks, especially if the `similarity_top_k` setting is as restrictive as 2. Conversely, a chunk size of 512 is likely to encompass all necessary information within the top chunks. To navigate this, we employ the **Faithfulness** and **Relevancy** metrics, which measure the absence of 'hallucinations' and the relevancy of responses with respect to the query and retrieved contexts.
2. **Response Generation Time**: As the `chunk_size` increases, so does the volume of information directed into the LLM. While this ensures more comprehensive context, it can also slow the system down.

In essence, determining the optimal `chunk_size` is about striking a balance: capturing all essential information without sacrificing speed.

## **Setup**

All dependencies are listed in `requirements.txt` (`llama-index`, `pypdf`, `python-dotenv`, `nest-asyncio`). Install them with `pip install -r requirements.txt` before running this notebook.

In [2]:
import nest_asyncio

nest_asyncio.apply()

import os
import time

from dotenv import load_dotenv

# Modern (0.10+) import paths — everything lives under llama_index.core
from llama_index.core import SimpleDirectoryReader, VectorStoreIndex, Settings
from llama_index.core.node_parser import SentenceSplitter
from llama_index.core.evaluation import (
    DatasetGenerator,
    FaithfulnessEvaluator,
    RelevancyEvaluator,
)
# LLM and Embeddings
from llama_index.llms.azure_openai import AzureOpenAI
from llama_index.embeddings.azure_openai import AzureOpenAIEmbedding

load_dotenv()

if not os.getenv("AZURE_OPENAI_API_KEY"):
    raise ValueError(
        "AZURE_OPENAI_API_KEY not found. Create a .env file in the project root "
        "with OPENAI_API_KEY=your_key_here"
    )

print("✅ Imports successful!")

✅ Imports successful!


## **Download Data**

We'll use the Uber 10-K SEC Filing for 2021 for this experiment.

In [5]:
# Create directory
os.makedirs("data/10k", exist_ok=True)

# Download PDF
import requests

url = "https://raw.githubusercontent.com/jerryjliu/llama_index/main/docs/examples/data/10k/uber_2021.pdf"

response = requests.get(url)
response.raise_for_status()

with open("data/10k/uber_2021.pdf", "wb") as f:
    f.write(response.content)

# Verify downloaded file
file_path = "data/10k/uber_2021.pdf"

print("File exists:", os.path.exists(file_path))
print("File size:", round(os.path.getsize(file_path) / 1024, 2), "KB")

File exists: True
File size: 1836.41 KB


## **Load Data**

Let's load the document.

In [6]:
# Load Data
reader = SimpleDirectoryReader("./data/10k/")
documents = reader.load_data()
print(f"Loaded {len(documents)} document(s), {sum(len(d.text) for d in documents):,} chars")

Loaded 307 document(s), 1,280,477 chars


## **Question Generation**

To select the right `chunk_size`, we'll compute Average Response Time, Faithfulness, and Relevancy for various sizes. The `DatasetGenerator` generates questions from the documents. We use a cheap, fast model (`gpt-6-luna`) for question generation since volume matters more than quality here.

In [7]:
# Generate 40 questions from the first 20 pages
eval_documents = documents[:20]

question_gen_llm =AzureOpenAI(
    engine=os.environ["AZURE_OPENAI_DEPLOYMENT_NAME"],
    model="gpt-4.1-mini",
    azure_endpoint=os.environ["AZURE_OPENAI_ENDPOINT"],
    api_key=os.environ["AZURE_OPENAI_API_KEY"],
    api_version="2024-10-21",
    temperature=0,
)

# Build nodes first, then construct the generator directly.
# (DatasetGenerator.from_documents internally applies a KeywordNodePostprocessor
# that requires an optional spacy install — constructing from nodes avoids it.)
question_gen_splitter = SentenceSplitter(chunk_size=512)
question_gen_nodes = question_gen_splitter.get_nodes_from_documents(eval_documents)

data_generator = DatasetGenerator(
    nodes=question_gen_nodes,
    llm=question_gen_llm,
)
eval_questions = data_generator.generate_questions_from_nodes(num=40)

print(f"✅ Generated {len(eval_questions)} evaluation questions")
print(f"   Example: {eval_questions[0]}")

C:\Users\bsmun\AppData\Local\Temp\ipykernel_19612\2689030010.py:19: DeprecationWarning: Call to deprecated class DatasetGenerator. (Deprecated in favor of `RagDatasetGenerator` which should be used instead.)
  data_generator = DatasetGenerator(
2026-11-22 07:17:11,431 - INFO - HTTP Request: POST https://moham-mi3bbwto-eastus2.cognitiveservices.azure.com/openai/deployments/gpt-4o/chat/completions?api-version=2024-10-21 "HTTP/1.1 200 OK"
2026-11-22 07:17:11,645 - INFO - HTTP Request: POST https://moham-mi3bbwto-eastus2.cognitiveservices.azure.com/openai/deployments/gpt-4o/chat/completions?api-version=2024-10-21 "HTTP/1.1 200 OK"
2026-11-22 07:17:11,716 - INFO - HTTP Request: POST https://moham-mi3bbwto-eastus2.cognitiveservices.azure.com/openai/deployments/gpt-4o/chat/completions?api-version=2024-10-21 "HTTP/1.1 200 OK"
2026-11-22 07:17:11,730 - INFO - HTTP Request: POST https://moham-mi3bbwto-eastus2.cognitiveservices.azure.com/openai/deployments/gpt-4o/chat/completions?api-version=2024

✅ Generated 40 evaluation questions
   Example: What is the official name of the document filed by Uber Technologies, Inc. for the fiscal year ended December 31, 2021?


f:\llama-index-tutorials\.venv\Lib\site-packages\llama_index\core\evaluation\dataset_generation.py:297: DeprecationWarning: Call to deprecated class QueryResponseDataset. (Deprecated in favor of `LabelledRagDataset` which should be used instead.)
  return QueryResponseDataset(queries=queries, responses=responses_dict)


In [8]:
# Strong model as the evaluation judge (temperature 0 for consistency)
eval_llm = AzureOpenAI(
    engine=os.environ["AZURE_OPENAI_DEPLOYMENT_NAME"],
    model="gpt-4.1-mini",
    azure_endpoint=os.environ["AZURE_OPENAI_ENDPOINT"],
    api_key=os.environ["AZURE_OPENAI_API_KEY"],
    api_version="2024-10-21",
    temperature=0,
)


faithfulness_evaluator = FaithfulnessEvaluator(llm=eval_llm)
relevancy_evaluator = RelevancyEvaluator(llm=eval_llm)

print("✅ Evaluators initialized (judge: gpt-6.1-sol)")

✅ Evaluators initialized (judge: gpt-6.1-sol)


## **Response Evaluation For A Chunk Size**

We evaluate each `chunk_size` on 3 metrics: Average Response Time, Average Faithfulness, and Average Relevancy. The function below handles node construction, index building, querying, and metric calculation.

In [11]:
def evaluate_response_time_and_accuracy(chunk_size, eval_questions):
    """Evaluate average response time, faithfulness, and relevancy for a chunk size.

    Responses are generated by gpt-5-nano and judged by gpt-5-nano.
    """
    total_response_time = 0
    total_faithfulness = 0
    total_relevancy = 0

    # Configure the response-generation model (the judge evaluators keep their own LLM)
    Settings.llm = AzureOpenAI(
        engine=os.environ["AZURE_OPENAI_DEPLOYMENT_NAME"],
        model="gpt-4.1-mini",
        azure_endpoint=os.environ["AZURE_OPENAI_ENDPOINT"],
        api_key=os.environ["AZURE_OPENAI_API_KEY"],
        api_version="2024-10-21",
        temperature=0,
    )
    # Configure Embedding Model
    Settings.embed_model = AzureOpenAIEmbedding(
        model="text-embedding-3-small",
        deployment_name=os.environ["AZURE_OPENAI_EMBEDDING_DEPLOYMENT_NAME"],
        azure_endpoint=os.environ["AZURE_OPENAI_ENDPOINT"],
        api_key=os.environ["AZURE_OPENAI_API_KEY"],
        api_version="2024-10-21",
    )
    Settings.chunk_size = chunk_size

    # Build nodes with the target chunk size, then index them
    # Overlap must be smaller than chunk_size (20% of chunk, capped at 200)
    splitter = SentenceSplitter(chunk_size=chunk_size, chunk_overlap=min(200, chunk_size // 5))
    nodes = splitter.get_nodes_from_documents(eval_documents)
    vector_index = VectorStoreIndex(nodes)

    # By default, similarity_top_k is 2 — pass it to as_query_engine() to experiment
    query_engine = vector_index.as_query_engine()

    num_questions = len(eval_questions)
    for question in eval_questions:
        start_time = time.time()
        response_vector = query_engine.query(question)
        elapsed_time = time.time() - start_time

        faithfulness_result = faithfulness_evaluator.evaluate_response(
            response=response_vector
        ).passing

        relevancy_result = relevancy_evaluator.evaluate_response(
            query=question, response=response_vector
        ).passing

        total_response_time += elapsed_time
        total_faithfulness += faithfulness_result
        total_relevancy += relevancy_result

    average_response_time = total_response_time / num_questions
    average_faithfulness = total_faithfulness / num_questions
    average_relevancy = total_relevancy / num_questions

    return average_response_time, average_faithfulness, average_relevancy

## **Testing Across Different Chunk Sizes**

We evaluate a range of chunk sizes to find the best metrics. (For large question sets, wrap the evaluators in `BatchEvalRunner` from `llama_index.core.evaluation` to run evaluations concurrently.)

In [19]:
async def evaluate_response_time_and_accuracy(chunk_size, eval_questions):
    if not eval_questions:
        raise ValueError("eval_questions must not be empty")

    Settings.llm = AzureOpenAI(
        engine=os.environ["AZURE_OPENAI_DEPLOYMENT_NAME"],
        model="gpt-4.1-mini",
        azure_endpoint=os.environ["AZURE_OPENAI_ENDPOINT"],
        api_key=os.environ["AZURE_OPENAI_API_KEY"],
        api_version="2024-10-21",
        temperature=0,
    )

    Settings.embed_model = AzureOpenAIEmbedding(
        model="text-embedding-3-small",
        deployment_name=os.environ["AZURE_OPENAI_EMBEDDING_DEPLOYMENT_NAME"],
        azure_endpoint=os.environ["AZURE_OPENAI_ENDPOINT"],
        api_key=os.environ["AZURE_OPENAI_API_KEY"],
        api_version="2024-10-21",
    )

    Settings.chunk_size = chunk_size
    splitter = SentenceSplitter(
        chunk_size=chunk_size,
        chunk_overlap=min(200, chunk_size // 5),
    )
    nodes = splitter.get_nodes_from_documents(eval_documents)
    vector_index = VectorStoreIndex(nodes)
    query_engine = vector_index.as_query_engine()

    total_response_time = 0.0
    total_faithfulness = 0
    total_relevancy = 0

    for question in eval_questions:
        start_time = time.perf_counter()
        response = await query_engine.aquery(question)
        total_response_time += time.perf_counter() - start_time

        faithfulness = await faithfulness_evaluator.aevaluate_response(
            response=response
        )
        relevancy = await relevancy_evaluator.aevaluate_response(
            query=question,
            response=response,
        )

        total_faithfulness += faithfulness.passing
        total_relevancy += relevancy.passing

    count = len(eval_questions)
    return (
        total_response_time / count,
        total_faithfulness / count,
        total_relevancy / count,
    )

In [21]:
import pandas as pd
from IPython.display import display

results = []

for chunk_size in [128, 256, 512, 1024, 2048]:
    avg_response_time, avg_faithfulness, avg_relevancy = (
        await evaluate_response_time_and_accuracy(chunk_size, eval_questions)
    )

    results.append({
        "Chunk Size": chunk_size,
        "Average Response Time (s)": avg_response_time,
        "Faithfulness": avg_faithfulness,
        "Relevancy": avg_relevancy,
    })

results_df = pd.DataFrame(results)
display(
    results_df.style.format({
        "Average Response Time (s)": "{:.2f}",
        "Faithfulness": "{:.3f}",
        "Relevancy": "{:.3f}",
    })
)

2026-11-22 08:34:06,703 - INFO - HTTP Request: POST https://moham-mi3bbwto-eastus2.cognitiveservices.azure.com/openai/deployments/text-embedding-ada-002/embeddings?api-version=2024-10-21 "HTTP/1.1 200 OK"
2026-11-22 08:34:07,527 - INFO - HTTP Request: POST https://moham-mi3bbwto-eastus2.cognitiveservices.azure.com/openai/deployments/text-embedding-ada-002/embeddings?api-version=2024-10-21 "HTTP/1.1 200 OK"
2026-11-22 08:34:08,117 - INFO - HTTP Request: POST https://moham-mi3bbwto-eastus2.cognitiveservices.azure.com/openai/deployments/text-embedding-ada-002/embeddings?api-version=2024-10-21 "HTTP/1.1 200 OK"
2026-11-22 08:34:09,113 - INFO - HTTP Request: POST https://moham-mi3bbwto-eastus2.cognitiveservices.azure.com/openai/deployments/text-embedding-ada-002/embeddings?api-version=2024-10-21 "HTTP/1.1 200 OK"
2026-11-22 08:34:09,549 - INFO - HTTP Request: POST https://moham-mi3bbwto-eastus2.cognitiveservices.azure.com/openai/deployments/text-embedding-ada-002/embeddings?api-version=2024

,Chunk Size,Average Response Time (s),Faithfulness,Relevancy
0,128,1.62,0.850,0.800
1,256,1.60,0.775,0.750
2,512,1.53,0.925,0.925
3,1024,1.59,0.925,0.925
4,2048,1.50,0.925,0.950
